# Module 21 — The GIL, Threads, and Processes

## Exercise 21.1 — Measure the GIL

Four workloads, three execution strategies. Predict all twelve results BEFORE
running. The predictions are the exercise; the numbers merely grade them.
Run:  python ex01_gil_lab.py

---

**How to work through this.** Each task below is its own cell. Run them one at a
time and read the output before moving on; that is the whole advantage of a
notebook over a script. Where a cell asks for a prediction, write it before you
run anything. Being wrong on purpose in a place where it costs nothing is how
the correct model gets built.

---

# The concepts behind this exercise

Read this before the tasks. Every idea the tasks below use is explained here, so
you should not need to leave this notebook.

The code cells in this part are demonstrations rather than exercises. Run them,
change a value, run them again. That is the whole point of having them here
instead of in a document.

## Concept 1. What the GIL actually is

A single mutex in the CPython interpreter. A thread must hold it to execute
bytecode. It is released:

- every 5 milliseconds by default (`sys.setswitchinterval`), so other threads
  get a turn;
- around **every blocking I/O call** — file reads, socket operations, `sleep`;
- inside C extensions that explicitly release it (NumPy's array operations,
  `hashlib`, `zlib`, `lxml`, database drivers).

So the accurate statement is: **CPython cannot run pure-Python CPU work in
parallel across threads.** Everything else is available.

In [ ]:
import sys
sys.getswitchinterval()          # 0.005

**Why it exists:** it makes reference counting (Module 02) safe without a lock
on every object, which makes single-threaded code fast and C extensions simple
to write. Removing it has been attempted repeatedly since 1999; every previous
attempt made single-threaded code substantially slower.

**PEP 703 and free-threading.** Python 3.13 ships an optional build
(`python3.13t`) with no GIL, using biased reference counting and per-object
locks. It is real and it works. The caveats: single-threaded code is currently
somewhat slower, most C extensions need updating, and it is not the default. By
3.15 or so this section may need rewriting — but the *decision framework* below
does not change, because it is about the nature of the work, not the
interpreter.

---

## Concept 3. Threads

In [ ]:
from threading import Thread, Lock, RLock, Event, Condition, Semaphore, local
import queue

def worker(n: int, results: list[int]) -> None:
    results.append(n * 2)

threads = [Thread(target=worker, args=(i, results)) for i in range(4)]
for t in threads: t.start()
for t in threads: t.join()          # ALWAYS join, or the program may exit first

Prefer `concurrent.futures` (section 5) to raw threads for almost everything.

### Race conditions

In [ ]:
counter = 0

def increment() -> None:
    global counter
    for _ in range(100_000):
        counter += 1          # NOT atomic: LOAD, ADD, STORE

Run this on eight threads and the result is not 800,000. `counter += 1` is
three bytecode operations (Module 24 shows them), and a thread switch between
the LOAD and the STORE loses an increment.

**What is atomic in CPython?** A single bytecode operation. `list.append`,
`dict[k] = v`, and `x = y` are atomic *as a consequence of the GIL* — an
implementation detail you should not build on. Anything that reads then writes
is not.

In [ ]:
lock = Lock()

def increment() -> None:
    global counter
    for _ in range(100_000):
        with lock:                 # always `with`, never acquire/release
            counter += 1

### The primitives

| Primitive | For |
|---|---|
| `Lock` | Mutual exclusion. Cannot be re-acquired by the same thread. |
| `RLock` | Re-entrant: the same thread may acquire it repeatedly |
| `Event` | One-shot signalling: "the thing has happened" |
| `Condition` | Wait for a predicate to become true |
| `Semaphore` | Limit concurrency to N |
| `Barrier` | Wait until N threads arrive |
| `local()` | Per-thread storage |
| `queue.Queue` | **A thread-safe queue. Use this instead of shared state.** |

**The best concurrency primitive is not sharing state at all.** A `queue.Queue`
between a producer and consumers eliminates the entire class of race condition,
because only the queue is shared and it is already correct.

### Deadlock

In [ ]:
# thread 1: with lock_a: with lock_b: ...
# thread 2: with lock_b: with lock_a: ...      -> deadlock

Two rules that prevent essentially all of it: **acquire locks in a consistent
global order**, and **use `timeout=` so a deadlock becomes an error rather than
a hang**.

---

## Concept 4. Processes

In [ ]:
from multiprocessing import Process, Pool, Queue, Value, Array, shared_memory

with Pool(processes=4) as pool:
    results = pool.map(cpu_heavy, items)

Separate memory, separate interpreters, separate GILs — so true parallelism, at
a price:

- **Everything crossing the boundary is pickled** (Module 19). Lambdas, closures,
  open files, locks and database connections cannot be sent.
- **Startup costs milliseconds.** `fork` is fast; `spawn` (the default on macOS
  and Windows, and on Linux from 3.14) re-imports your module in the child.
- **Under `spawn`, module-level code runs again in every child.** Without an
  `if __name__ == "__main__":` guard (Module 01), you get a process bomb.

In [ ]:
if __name__ == "__main__":          # MANDATORY with spawn
    with Pool() as pool: ...

For large data, `multiprocessing.shared_memory` avoids the copy entirely — and
is what `numpy` users reach for when the array is bigger than the pickle budget.

---

## Concept 5. `concurrent.futures`: use this

One API, two backends, and swapping between them is one word.

In [ ]:
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor, as_completed

with ThreadPoolExecutor(max_workers=8) as ex:        # I/O bound
    futures = {ex.submit(fetch, url): url for url in urls}
    for future in as_completed(futures):
        url = futures[future]
        try:
            data = future.result()          # re-raises the worker's exception
        except Exception:
            logger.exception("failed: %s", url)

with ProcessPoolExecutor() as ex:                    # CPU bound
    results = list(ex.map(cpu_heavy, items))

Three things worth knowing:

**`executor.map` returns results in order and re-raises on iteration;
`as_completed` yields futures as they finish.** Use `map` when you want the
results in order, `as_completed` when you want to handle each as it arrives.

**Exceptions are stored in the future**, not raised in the worker. If you never
call `.result()`, the exception disappears silently — the most common
`concurrent.futures` bug.

**`max_workers` defaults** to `min(32, cpu_count + 4)` for threads and
`cpu_count` for processes. For I/O-bound work the right number is usually far
higher than the CPU count and is found by measuring.

---

---

# Now the exercise

You have everything you need. Work top to bottom, and where a cell asks for a
prediction, write it before you run anything.

## The concepts this exercise uses

These are the numbered sections of [the module README](../README.md). If a task below stops making sense, the section named next to it is the one to re-read.

- Section 1: What the GIL actually is
- Section 2: The decision, first
- Section 3: Threads
- Section 4: Processes
- Section 5: `concurrent.futures`: use this

> The teaching for this module currently lives in the README rather than in this notebook. Read it alongside these cells.

## Setup

Run this first. It is the imports and any shared values the tasks below need.

In [ ]:
from __future__ import annotations

import hashlib
import math
import time
import urllib.request
from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor

N_TASKS = 8

PREDICTIONS = """
Fill this in BEFORE running. Write the expected speedup versus serial.

                        | serial | threads | processes
  pure-Python CPU       |  1.0x  |         |
  hashlib (C, releases) |  1.0x  |         |
  time.sleep (fake I/O) |  1.0x  |         |
  file reads (real I/O) |  1.0x  |         |

Then, for each cell, write ONE WORD explaining why: "GIL", "released",
"waiting", "pickling", "startup".
"""

---

## `cpu_pure`

Pure Python arithmetic. The GIL applies fully.

In [ ]:
def cpu_pure(n: int) -> float:
    """Pure Python arithmetic. The GIL applies fully."""
    total = 0.0
    for i in range(1, 2_000_000):   # tune this until serial takes ~2s
        total += math.sqrt(i) / (i + n)
    return total

---

## `cpu_c_extension`

hashlib releases the GIL for large inputs.

In [ ]:
def cpu_c_extension(n: int) -> str:
    """hashlib releases the GIL for large inputs."""
    data = bytes(2_000_000)
    h = hashlib.sha256()
    for _ in range(4):
        h.update(data)
    return h.hexdigest()

---

## `fake_io`

time.sleep releases the GIL. This is the cleanest demonstration.

In [ ]:
def fake_io(n: int) -> int:
    """time.sleep releases the GIL. This is the cleanest demonstration."""
    time.sleep(0.25)
    return n

---

## `real_io`

Reading a file. Adjust the path if needed.

In [ ]:
def real_io(n: int) -> int:
    """Reading a file. Adjust the path if needed."""
    total = 0
    with open("/usr/share/dict/words" if _has_words() else __file__, "rb") as fh:
        for _ in range(20):
            fh.seek(0)
            total += len(fh.read())
    return total

---

## `_has_words`

_ has words_

In [ ]:
def _has_words() -> bool:
    import os
    return os.path.exists("/usr/share/dict/words")

---

## `run_serial`

_run serial_

In [ ]:
def run_serial(fn, tasks):  # type: ignore[no-untyped-def]
    return [fn(i) for i in tasks]

---

## `run_threads`

_run threads_

In [ ]:
def run_threads(fn, tasks):  # type: ignore[no-untyped-def]
    with ThreadPoolExecutor(max_workers=len(tasks)) as ex:
        return list(ex.map(fn, tasks))

---

## `run_processes`

_run processes_

In [ ]:
def run_processes(fn, tasks):  # type: ignore[no-untyped-def]
    with ProcessPoolExecutor(max_workers=len(tasks)) as ex:
        return list(ex.map(fn, tasks))

---

## `measure`

_measure_

In [ ]:
def measure(label: str, fn) -> None:  # type: ignore[no-untyped-def]
    tasks = list(range(N_TASKS))
    times: dict[str, float] = {}
    for name, runner in [("serial", run_serial), ("threads", run_threads),
                         ("processes", run_processes)]:
        start = time.perf_counter()
        runner(fn, tasks)
        times[name] = time.perf_counter() - start

    base = times["serial"]
    print(f"\n  {label}")
    for name, elapsed in times.items():
        print(f"    {name:<10} {elapsed:6.2f}s   {base / elapsed:5.2f}x")

---

## Run it

This is what running the original file did. Everything above must have been run first.

In [ ]:
# TODO 1: run it, and compare against your predictions. How many did you get?

# TODO 2: explain the hashlib row. Look at CPython's Modules/_hashopenssl.c or
#         the docs -- above what input size does hashlib release the GIL, and
#         why is there a threshold at all rather than always releasing?

# TODO 3: the processes column for fake_io is probably NOT 8x. Explain the gap.
#         Then reduce the sleep to 0.001 and re-run. What happened to the
#         process speedup, and what does that tell you about when process
#         overhead dominates?

# TODO 4: sys.setswitchinterval(0.000001) and re-run the pure-Python CPU row
#         with threads. Predict first: faster, slower, or the same? Explain the
#         result in terms of what a switch costs.

# TODO 5: write a FIFTH workload that is half CPU and half I/O, and find the
#         strategy that wins. Then explain why "it depends" is the correct
#         answer to "should I use threads or processes".


if __name__ == "__main__":
    print(PREDICTIONS)
    print(f"running {N_TASKS} tasks per workload...")
    measure("pure-Python CPU", cpu_pure)
    measure("hashlib (C extension)", cpu_c_extension)
    measure("time.sleep (fake I/O)", fake_io)
    measure("file reads (real I/O)", real_io)

---

## Before you move on

- [ ] Every cell above ran, in order, on a fresh kernel.
- [ ] You wrote a prediction before running, wherever one was asked for.
- [ ] You can say in one sentence what each task was actually testing.
- [ ] Anything that surprised you is written down in `PROGRESS.md`.

Compare against the worked answers in `../solutions/` only after your own
attempt runs.